# Lekcia 01 - Úvod do AI agentov

Vitajte v prvej lekcii kurzu **AI agenti pre začiatočníkov**!

**AI agent** je program, ktorý používa veľký jazykový model (LLM) ako svoj rozmýšľací engine a môže v reálnom svete vykonávať *akcie* — volať API, dotazovať databázy alebo spúšťať kód — aby dosiahol cieľ v mene používateľa.

V tomto notebooku si vytvoríte svojho prvého agenta: **Cestovného agenta**, ktorý odporúča dovolenkové destinácie. Počas toho sa naučíte:

1. Pripojiť sa k Azure AI Foundry Agent Service pomocou **Microsoft Agent Frameworku**.
2. Dodať agentovi **nástroj** — jednoduchú Python funkciu, ktorú môže volať.
3. Spustiť agenta a skontrolovať jeho odpoveď.
4. Streamovať odpoveď agenta token po tokene.


## Nastavenie

Pred spustením tohto notebooku sa uistite, že máte:

1. **Projekt Azure AI Foundry** s nasadeným chatovacím modelom (napr. `gpt-4o-mini`).
2. **Prihlásený cez Azure CLI** — spustite `az login` vo vašom termináli.
3. **Nastavené požadované premenné prostredia:**
   - `AZURE_AI_PROJECT_ENDPOINT` — endpoint vášho projektu Azure AI Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — názov vášho nasadeného modelu.

Nasledujúca bunka nainštaluje Python balíčky, ktoré potrebujete.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Vytvorenie vášho prvého agenta

Agent potrebuje dve veci:

- **Inštrukcie**, ktoré mu povedia *kto je* a *ako sa má správať* (systémový prompt).
- **Nástroje** — Python funkcie označené dekorátorom `@tool`, ktoré môže agent volať na získavanie informácií alebo vykonávanie akcií.

Nižšie definujeme jednoduchý nástroj, ktorý vracia zoznam populárnych dovolenkových destinácií. Agent bude tento nástroj využívať, keď používateľ požiada o odporúčania na cestovanie.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Prúdové odpovede

Pre interaktívnejší zážitok môžete **prúdiť** odpoveď agenta. Namiesto čakania na celú odpoveď agent postupne odovzdáva textové časti, ako sú generované. Toto je obzvlášť užitočné v chatovacích rozhraniach, kde chcete zobrazovať výstup v reálnom čase.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

V tejto lekcii ste sa naučili, ako:

- **Vytvoriť poskytovateľa** ktorý sa pripája k Azure AI Foundry Agent Service cez `AzureAIProjectAgentProvider`.
- **Definovať nástroj** pomocou dekorátora `@tool`, aby agent mohol volať vaše Python funkcie.
- **Spustiť agenta** so správou od používateľa a vypísať jeho odpoveď.
- **Streamovať odpovede** pre výstup v reálnom čase.

V nasledujúcej lekcii preskúmame agentové rámce podrobnejšie a naučíme sa, ako dať agentom výkonnejšie nástroje a schopnosti viacstupňového uvažovania.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Upozornenie**:  
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, berte prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Originálny dokument v jeho pôvodnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre dôležité informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za akékoľvek nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
